# Airborne Magnetics Inversion — Equivalent Source + Full 3D

**Prerequisite:** run `prepare_data.ipynb` first to generate `synthetic_survey.msgpack`.

This notebook demonstrates the two-stage workflow:

1. **Equivalent source inversion** — fits flight-line TMI with a flat dipole layer,
   then predicts Bx / By / Bz / TMI on a regular grid.
2. **Full 3D inversion** — recovers a 3D susceptibility model from the gridded fields.

## Setup

```bash
pip install -e deps/mag_inversion/
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from AirMagTools.magdata import MagData
from mag_inversion import MagEquivalentSourceSystem, MagInversion3DSystem

## 1  Load survey data

In [ ]:
mag_data = MagData.load('synthetic_survey.msgpack')
print(mag_data)

In [ ]:
# Quick map of the raw TMI data
df = mag_data.data.reset_index()

# Plot in local coords (subtract origin so axes are readable)
e0 = df.easting.mean()
n0 = df.northing.mean()

plt.figure(figsize=(7, 6))
sc = plt.scatter(df.easting - e0, df.northing - n0,
                 c=df.magcom, s=2, cmap='RdBu_r')
plt.colorbar(sc, label='magcom (nT)')
plt.title('Observed TMI')
plt.xlabel('Easting offset (m)'); plt.ylabel('Northing offset (m)')
plt.axis('equal'); plt.tight_layout(); plt.show()

## 2  Equivalent Source Inversion

The `MagEquivalentSourceSystem` reads Earth field parameters directly from
`mag_data.meta`, so the only thing we need to configure is the mesh geometry
and inversion settings.

In [ ]:
equiv = MagEquivalentSourceSystem(
    mag_data,
    # Mesh
    layer__cell_size=50,           # 50 m horizontal cell size
    layer__depth_below_flight=30,  # layer 30 m below minimum flight altitude
    layer__padding_cells=6,
    # Inversion
    regularization__alpha_s=1e-4,
    optimizer__max_iter=20,
    # Sensitivity storage — 'ram' is fine for this survey size
    store_sensitivities='ram',
    # Output grid
    output__components=['tmi', 'bx', 'by', 'bz'],
    output__xy_spacing=50,
)

In [ ]:
model_equiv, mesh_equiv, sim_equiv = equiv.invert()

In [ ]:
ds_equiv = equiv.to_xarray(model_equiv, mesh_equiv, sim_equiv)
print(ds_equiv)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, comp in zip(axes, ['tmi', 'bx', 'by', 'bz']):
    ds_equiv[comp].plot(ax=ax, cmap='RdBu_r')
    ax.set_title(comp.upper())
    ax.set_aspect('equal')
plt.suptitle('Equivalent source — gridded output', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# Save for downstream use or visualisation in Nagelfluh
# ds_equiv.webxtile.to_webxtile('equiv_source_output/')

## 3  Full 3D Inversion

Pass the equivalent source xarray Dataset directly to `MagInversion3DSystem`.
Field parameters, CRS and output altitude are carried in the dataset attributes
and do not need to be repeated.

In [ ]:
inv3d = MagInversion3DSystem(
    ds_equiv,
    # Model type: 'scalar' (induced only), 'vector' (MVI), or 'amplitude'
    model_type='scalar',
    # Mesh
    mesh__core_cell_size=50,
    mesh__depth_core=500,
    mesh__max_distance=2000,
    # Inversion
    regularization__alpha_s=1e-4,
    optimizer__max_iter=15,
    directives__sensitivity_weights__enable=True,
    # Sensitivity storage
    store_sensitivities='ram',
)

In [ ]:
model_3d, mesh_3d, sim_3d, active_3d = inv3d.invert()

In [ ]:
ds_3d = inv3d.to_xarray(model_3d, mesh_3d, active_3d)
print(ds_3d)

In [ ]:
chi = ds_3d['susceptibility']

# Horizontal slice nearest to z = –150 m (centre of true prism)
z_idx = int(np.argmin(np.abs(chi.z.values - (-150))))
# Vertical section nearest to y = survey-origin northing
y_mid = float(ds_equiv.y.mean())
y_idx = int(np.argmin(np.abs(chi.y.values - y_mid)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

chi.isel(z=z_idx).plot(ax=ax1, cmap='viridis')
ax1.set_title(f'Recovered χ — horizontal slice  z ≈ {chi.z.values[z_idx]:.0f} m')
ax1.set_aspect('equal')

chi.isel(y=y_idx).plot(ax=ax2, cmap='viridis', yincrease=False)
ax2.set_title(f'Recovered χ — vertical section  y ≈ {chi.y.values[y_idx]:.0f} m')

plt.tight_layout(); plt.show()

In [ ]:
# Save 3D result for Nagelfluh visualisation
# ds_3d.webxtile.to_webxtile('mag3d_output/')